In [1]:
import requests
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def create_embedding(text_list):
    r = requests.post("http://localhost:11434/api/embed", json={
        "model": "bge-m3",
        "input": text_list
    })

    embedding = r.json()["embeddings"]
    return embedding

In [3]:
def inference(prompt):
    r = requests.post("http://localhost:11434/api/generate", json={
        # "model": "deepseek-r1",
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
    })

    response = r.json()
    print(response)
    return response

In [4]:
df = joblib.load("../data/embeddings/chunk_embeddings.joblib")

In [5]:
incoming_query = input("Ask a Question: ")
question_emb = create_embedding([incoming_query])[0]
# print(question_emb)

similarities = cosine_similarity(np.vstack(df["embedding"]), [question_emb]).flatten()

top_result = 5
max_index = similarities.argsort()[::-1][0:top_result]
print(max_index)
new_df = df.loc[max_index]
new_df

Ask a Question:  where is tailwind css taught


[ 0 14 33 23 34]


,number,title,start,end,text,chunk_id,embedding
0,100,Exercise 17 - Design This Layout,0.00,2.50,"Guys, I told you about Tailwind CSS.",0,"[-0.05321666, 0.002405325, -0.050752055, 0.010..."
14,100,Exercise 17 - Design This Layout,52.38,56.04,You have to use Tailwind CSS to design a layout.,14,"[-0.058305018, 0.008270422, -0.03620454, -0.00..."
33,100,Exercise 17 - Design This Layout,115.30,119.30,And this play.tailwindcss.com which I told yo...,33,"[-0.034609213, 0.0060682953, -0.0648977, 0.004..."
23,100,Exercise 17 - Design This Layout,77.80,80.46,But you also have to do it with the help of T...,23,"[-0.062160436, 0.032005697, -0.026386674, 0.00..."
34,100,Exercise 17 - Design This Layout,119.30,123.30,don't think that you have become very smart b...,34,"[-0.05428051, 0.0037771754, -0.05885777, -0.00..."


In [6]:
for idx, item in new_df.iterrows():
    print(idx, item["title"], item["number"], item["text"], item["start"], item["end"])

0 Exercise 17 - Design This Layout  100  Guys, I told you about Tailwind CSS. 0.0 2.5
14 Exercise 17 - Design This Layout  100  You have to use Tailwind CSS to design a layout. 52.379999999999995 56.04
33 Exercise 17 - Design This Layout  100  And this play.tailwindcss.com which I told you in the last video, 115.3 119.3
23 Exercise 17 - Design This Layout  100  But you also have to do it with the help of Tailwind CSS. 77.8 80.46000000000001
34 Exercise 17 - Design This Layout  100  don't think that you have become very smart by learning play.tailwindcss.com 119.3 123.3


## Creating prompt

In [7]:
prompt = f'''I am teaching web development in my sigma web development course. Here are the video chunks containing video title,
video number, start time in seconds, end time in seconds, the text at that time:

{new_df[["title", "number", "start", "end", "text"]].to_json(orient="records")}
--------------------------------------------------
"{incoming_query}"
User asked this question related to the video chunks, you have to
answer in a human way (dont't mention the above format, it's just for you) where and how much content is taught in which video (in which video and at what timestamp)
and guide the user to go to that particular video. If user asks unrelated question, tell him that you can only answer
questions related to the course.
'''

In [8]:
response = inference(prompt)["response"]
print(response)

with open("response.txt", "w") as f:
    f.write(response)

{'model': 'llama3.2', 'created_at': '2026-11-26T00:28:24.9346372Z', 'response': "In the Sigma Web Development course, Tailwind CSS is taught in Video 100, specifically in the following timestamps:\n\n* The video starts with an introduction to Tailwind CSS (0.0 seconds - 2.5 seconds).\n* Tailwind CSS is mentioned again at 52.38 seconds to 56.04 seconds, where the instructor emphasizes the importance of using it to design the layout.\n* The instructor also mentions playing.tailwindcss.com, which is likely a tutorial or resource website, at 115.3 seconds to 119.3 seconds.\n* Additionally, the instructor mentions using Tailwind CSS to design the layout, but also notes that it's not a substitute for learning from resources like play.tailwindcss.com, at 77.8 seconds to 80.46 seconds and 119.3 seconds to 123.3 seconds.\n\nIf you'd like to learn more about Tailwind CSS and how to use it for designing layouts, I recommend checking out Video 100.", 'done': True, 'done_reason': 'stop', 'context':